# SAC Irrigation Training - v2.18-P3b (alpha=0.002 + late noise reinjection)

Architecturally BYTE-IDENTICAL to v2.16/v2.17 (marker=2.16, RAIN_REF=30, linear r6, LayerNorm VDN critic). **Only training-time changes vs v2.17-P3:**
1. `ent_coef`: 0.005 -> **0.002** (weaken the entropy mu-pin further).
2. **Late noise reinjection:** exploration noise anneals 0.30 -> 0 over the first 60k (as v2.17), then a triangular pulse of peak 0.15 over [150k, 180k] (peaking at 165k), then back to 0.

## Why

v2.17-P3 (alpha=0.005 + decaying noise) succeeded but left a residual: wet x1 reached 144 (vs MPC 130 / auto-alpha 132) with 58% of cell-days still over FC. Two compounding causes: (a) the alpha=0.005 entropy spring still pulls mu toward the 6 mm tanh-center; (b) once noise hit 0 at 60k, the policy stopped sampling sub-2mm actions and the critic's data there went stale. v2.18-P3b attacks both: lower alpha (a), and a late pulse when the critic is well-trained (b).

| wet-year (mean of 3 budgets) | v2.16-fixed | v2.17-P3 | auto-alpha | MPC |
|---|---|---|---|---|
| x1 median (mm) | 152 | 144 | 132 | 130 |
| waterlog days | 76 | 54 | 35 | 18 |
| water (mm) | 397 | 362 | 324 | 309 |
| yield (kg/ha) | 3391 | 3604 | 3633 | 3752 |

## Acceptance (decide on x1/waterlog, NOT corr or yield)

PRIMARY: wet x1 median < 138 mm AND wet waterlog < 48 days.
SECONDARY: wet water < 350 mm; wet/100 u_mean clearly < dry/100 u_mean.
COVERAGE (now fixed - wet detected by seasonal rainfall >=120mm): `p3/frac_low_action_wet` > 0 in BOTH the initial decay phase AND the reinjection pulse.
STABILITY WATCH: q_pred_mean must never go negative; |q_inflation_pct| < 80%. **If breached, that is the signal to migrate to TD3 (target-policy smoothing replaces entropy's smoothing) - do NOT lower alpha further.**

## Colab note
Results persist to Google Drive via the Cell-1 mount + archive cell. Use T4 (~2-2.5h) or A100 (~30-55min).


In [ ]:
# Mount Drive (persistent storage).
from google.colab import drive; drive.mount('/content/drive')
import os; DRIVE_ROOT='/content/drive/MyDrive/thesis_v218_p3b_runs'; os.makedirs(DRIVE_ROOT,exist_ok=True)
print('Drive mounted:', DRIVE_ROOT)


In [ ]:
# Clone repo + install deps (SB3 pinned 2.6.0).
import subprocess, sys, os
WORK = '/content'
repo = os.path.join(WORK, 'thesis')
if os.path.exists(repo):
    subprocess.run(['rm', '-rf', repo], check=True)
subprocess.run(['git', 'clone', 'https://github.com/taratorbati/thesis.git', repo], check=True)
os.chdir(repo); sys.path.insert(0, repo)
subprocess.run(['pip', 'install', '--quiet',
                'stable-baselines3==2.6.0', 'gymnasium', 'wandb', 'pytest'], check=True)
import torch
print(f'PyTorch: {torch.__version__}  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# WandB + GPU.
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY']=userdata.get('WANDB_API_KEY'); print('OK WANDB key loaded.')
except Exception as e:
    try:
        import getpass; k=getpass.getpass('WANDB key (Enter to skip): ').strip()
        if k: os.environ['WANDB_API_KEY']=k; print('OK set.')
        else: print('Skipping WandB.')
    except Exception: print('Skipping WandB.')
import subprocess; print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout or 'no GPU')


In [ ]:
# Pre-flight: smoke tests + 1000-step pilot exercising the v2.18-P3b wiring
# (NormalActionNoise + LateNoiseReinjectionCallback + fixed coverage telemetry).
import subprocess, sys
print('Smoke tests...')
assert subprocess.run([sys.executable,'-m','pytest','tests/test_rl_smoke.py','-v','--tb=short']).returncode==0, 'SMOKE FAILED'
print('\nFactorized-critic tests...')
assert subprocess.run([sys.executable,'-m','pytest','tests/test_factorized_critic.py','-v','--tb=short']).returncode==0, 'CRITIC TESTS FAILED'
print('\n1000-step pilot (compresses the noise schedule so both phases run)...')
from src.rl.train_v218_p3b import train_sac_v218_p3b
_ = train_sac_v218_p3b(seed=999, output_dir='/content/pilot', wandb_project=None,
                       total_timesteps=1000, explore_decay_steps=300,
                       reinject_start=500, reinject_end=800)
print('\nOK pre-flight passed. Proceed.')


In [ ]:
# Full 250k training (SAC v2.18-P3b: alpha=0.002 + late reinjection).
# ~30-55 min A100 / ~2-2.5 h T4. Defaults set in train_v218_p3b.py.
SEED = 0    # CHANGE per session
from src.rl.train_v218_p3b import train_sac_v218_p3b
model = train_sac_v218_p3b(
    seed=SEED,
    output_dir='/content/thesis/results/rl',
    wandb_project='sac-irrigation-thesis',
    total_timesteps=250_000,
    ent_coef=0.002,                 # *** v2.18-P3b ***
    explore_sigma_start=0.30, explore_sigma_floor=0.0,
    explore_sigma_reinject=0.15,    # *** late pulse peak ***
    explore_decay_steps=60_000,
    reinject_start=150_000, reinject_end=180_000,
)
print('Training complete.')


In [ ]:
# Persist to Drive (CRITICAL).
import shutil, os, datetime
src=f'/content/thesis/results/rl/sac_v218_p3b_seed{SEED}'
dst=os.path.join(DRIVE_ROOT, f'sac_v218_p3b_seed{SEED}_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
shutil.copytree(src,dst,ignore=shutil.ignore_patterns('replay_buffer_latest.pkl'))
print('Archived to Drive:', dst)


In [ ]:
# Post-training 9-cell eval. exp_rl.py auto-dispatches on marker=2.16 and
# applies RAIN_REF=30. The Windows UTF-8 stdout fix in exp_rl.py prevents the
# UnicodeEncodeError on the alpha-containing checkpoint label.
# IMPORTANT: exp_rl.py does NOT accept --output-dir (that flag caused an
# argparse error earlier). It writes results to its default runs directory.
import subprocess, sys, os
model_path = f'/content/thesis/results/rl/sac_v218_p3b_seed{SEED}/best_model/best_model.zip'
final_path = f'/content/thesis/results/rl/sac_v218_p3b_seed{SEED}/sac_v218_p3b_seed{SEED}_final.zip'

print('Evaluating BEST checkpoint (perfect forecast)...')
r = subprocess.run([sys.executable,'-m','scripts.experiments.exp_rl',
    '--mode','eval','--model',model_path,
    '--scenario','all','--budget','all','--forecast','perfect'],
    capture_output=True, text=True)
print(r.stdout[-1500:]);
if r.returncode!=0: print('STDERR:', r.stderr[-2000:])
assert r.returncode==0, 'PERFECT EVAL FAILED'

print('\nEvaluating BEST checkpoint (noisy forecast, robustness)...')
subprocess.run([sys.executable,'-m','scripts.experiments.exp_rl',
    '--mode','eval','--model',model_path,
    '--scenario','all','--budget','all','--forecast','noisy','--noise-seed','42'],
    capture_output=False)

if os.path.exists(final_path):
    print('\nEvaluating FINAL (250k) checkpoint (perfect)...')
    subprocess.run([sys.executable,'-m','scripts.experiments.exp_rl',
        '--mode','eval','--model',final_path,
        '--scenario','all','--budget','all','--forecast','perfect'],
        capture_output=False)
print('\nNOTE: eval outputs are under results/runs/<model-name>/. Locate them in the next cell.')

# Copy eval outputs to Drive.
import shutil, datetime, os
dst=os.path.join(DRIVE_ROOT, 'eval_'+datetime.datetime.now().strftime('%Y%m%d_%H%M%S'))
shutil.copytree('/content/thesis/results/runs', dst, dirs_exist_ok=True)
print('Eval outputs archived to:', dst)


In [ ]:
# PRIMARY DIAGNOSTIC (decide on x1/waterlog). Auto-locate the eval output dir.
import pandas as pd, numpy as np, json, glob, os
cands = [d for d in glob.glob('/content/thesis/results/runs/*v218_p3b*') if os.path.isdir(d)]
cands += [d for d in glob.glob('/content/thesis/results/runs/*') if os.path.isdir(d) and
          glob.glob(os.path.join(d,'sac_perfect_det_wet*100*seed0.parquet'))]
OUTPUT_DIR = next(iter(sorted(set(cands), key=os.path.getmtime, reverse=True)), None)
print('Eval dir:', OUTPUT_DIR)
assert OUTPUT_DIR, 'No eval output dir with wet/100 parquet found - check the eval cell ran.'

def wet_agg(d):
    ys,ws,wls,x1s=[],[],[],[]
    for b in ['100pct','85pct','70pct']:
        pj=glob.glob(os.path.join(d,f'sac_perfect_det_wet*{b}*seed0.json'))
        pq=glob.glob(os.path.join(d,f'sac_perfect_det_wet*{b}*seed0.parquet'))
        if not pj or not pq: continue
        m=json.load(open(pj[0]))['final_metrics']; df=pd.read_parquet(pq[0])
        ys.append(m['yield_kg_ha']); ws.append(m.get('water_used_mm'))
        wls.append(m.get('waterlog_days_per_agent')); x1s.append(float(df['x1'].median()))
    f=lambda a:float(np.mean([v for v in a if v is not None])) if a else float('nan')
    return f(ys),f(ws),f(wls),f(x1s)

y,w,wl,x1 = wet_agg(OUTPUT_DIR)
print('='*64)
print(f'{"metric":<18s}{"v2.18":>8s}{"v2.17":>8s}{"autoA":>7s}{"MPC":>6s}{"target":>9s}')
print(f'{"x1 median":<18s}{x1:8.1f}{144:8.0f}{132:7.0f}{130:6.0f}{"< 138":>9s}')
print(f'{"waterlog days":<18s}{wl:8.1f}{54:8.0f}{35:7.0f}{18:6.0f}{"< 48":>9s}')
print(f'{"water (mm)":<18s}{w:8.0f}{362:8.0f}{324:7.0f}{309:6.0f}{"< 350":>9s}')
print(f'{"yield (kg/ha)":<18s}{y:8.0f}{3604:8.0f}{3633:7.0f}{3752:6.0f}{"(info)":>9s}')
print('\nPRIMARY:')
print(f'  x1 < 138 : {"PASS" if x1<138 else "FAIL"} ({x1:.1f})')
print(f'  wlog < 48: {"PASS" if wl<48 else "FAIL"} ({wl:.1f})')

def umean(d,s,b):
    pq=glob.glob(os.path.join(d,f'sac_perfect_det_{s}*{b}*seed0.parquet'))
    return float(pd.read_parquet(pq[0])['u'].mean()) if pq else float('nan')
du,wu=umean(OUTPUT_DIR,'dry','100pct'),umean(OUTPUT_DIR,'wet','100pct')
print(f'\n  dry/100 u_mean={du:.2f}  wet/100 u_mean={wu:.2f}  gap={du-wu:.2f} mm (v2.17 gap ~1.0)')


In [ ]:
# COVERAGE + SIGMA SCHEDULE DIAGNOSTIC (telemetry now fixed: wet by rainfall).
import pandas as pd, os, glob
import matplotlib.pyplot as plt
run_dir = f'/content/thesis/results/rl/sac_v218_p3b_seed{SEED}'
cov = os.path.join(run_dir,'low_action_coverage_log.csv')
sig = os.path.join(run_dir,'exploration_sigma_log.csv')
if os.path.exists(cov):
    c=pd.read_csv(cov)
    decay=c[c.step<=60000]; pulse=c[(c.step>=150000)&(c.step<=180000)]
    print('frac_low_action ALL  : peak', round(c['frac_low_action'].max(),3))
    if 'frac_low_action_wet' in c:
        wetc=c['frac_low_action_wet'].dropna()
        wd=decay['frac_low_action_wet'].dropna(); wp=pulse['frac_low_action_wet'].dropna()
        print(f'frac_low_action_WET  : non-null rows {len(wetc)}/{len(c)}  '
              f'(was 0 in v2.17 - telemetry fix worked if >0)')
        if len(wd): print(f'  decay-phase  wet: mean {wd.mean():.3f} max {wd.max():.3f}')
        if len(wp): print(f'  pulse-phase  wet: mean {wp.mean():.3f} max {wp.max():.3f}  '
                          f'<- repopulation check')
    fig,ax=plt.subplots(1,2,figsize=(13,4))
    ax[0].plot(c['step'],c['frac_low_action'],lw=1,label='all')
    if 'frac_low_action_wet' in c: ax[0].plot(c['step'],c['frac_low_action_wet'],'.',ms=3,label='wet')
    ax[0].axvspan(150000,180000,alpha=0.12,color='red',label='pulse')
    ax[0].set_title('frac actions < 1 mm/day'); ax[0].legend(); ax[0].grid(alpha=0.3)
    if os.path.exists(sig):
        s=pd.read_csv(sig); ax[1].plot(s['step'],s['sigma'],lw=1)
        ax[1].set_title('exploration sigma (decay + late pulse)'); ax[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print('No coverage log at', cov)


In [ ]:
# STABILITY: did alpha=0.002 + late pulse stay clean? (Decision gate for TD3.)
import os, glob
run_dir = f'/content/thesis/results/rl/sac_v218_p3b_seed{SEED}'
br = os.path.join(run_dir,'bias_ratio_log.csv')
if os.path.exists(br):
    import pandas as pd
    b=pd.read_csv(br)
    print(b.to_string(index=False))
    neg = (b['q_pred_mean']<0).any(); maxinf = b['q_inflation_pct'].abs().max()
    print(f'\n  q_pred_mean ever negative: {neg}  (must be False)')
    print(f'  max |q_inflation_pct|:     {maxinf:.1f}%  (must be < 80)')
    print(f'  STABILITY: {"CLEAN -> push lever / consider TD3 next" if (not neg and maxinf<80) else "BREACHED -> go TD3, do NOT lower alpha"}')
else:
    print('No bias_ratio_log.csv at', br)


In [ ]:
# Resume from checkpoint (if session died). Fill in the Drive run dir from the archive cell.
# SEED = 0; STEP = 100_000
# CKPT = f'/content/drive/MyDrive/thesis_v218_p3b_runs/<run-dir>/sac_v218_p3b_seed{SEED}/checkpoints/sac_v218_p3b_seed{SEED}_{STEP}_steps.zip'
# from src.rl.train_v212 import AsymmetricLRSAC
# from src.rl.networks import V216CTDESACPolicy
# model = AsymmetricLRSAC.load(CKPT, custom_objects={'policy_class': V216CTDESACPolicy})
# # model.learn(total_timesteps=..., reset_num_timesteps=False)
# # NOTE: action_noise is NOT restored. If resuming before step 180k, recreate
# # NormalActionNoise + LateNoiseReinjectionCallback so the schedule continues.
